In [2]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# Load the data
df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

# Define features and target
X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values
y = df['fc (MPa)'].values

# Scale the features and target using the same process as in training
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1))

# Convert the scaled features to torch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Define the model architecture
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Parameters for the trained model
input_dim = X_tensor.shape[1]  # Number of features
layers = 2  # Based on the model architecture used
neurons = 52  # Based on the model architecture used

# Instantiate the model
model = RegressionModel(input_dim=input_dim, layers=layers, neurons=neurons)

# Load the saved model weights
model.load_state_dict(torch.load('KINN_saved_models/model_run_10.pt'))
model.eval()  # Set the model to evaluation mode

# Make predictions
with torch.no_grad():
    fc_hat_scaled = model(X_tensor)  # Get predictions (scaled)

# Convert predictions back to the original scale using the inverse transformation
fc_hat = y_scaler.inverse_transform(fc_hat_scaled.numpy())

# Add predictions as the final column in the dataframe
df['fc_hat (MPa)'] = fc_hat

# Save the dataframe with predictions to a new Excel file, appending predictions as the last column
df.to_excel('updated_fc_predictions_with_fc_hat.xlsx', index=False)
print("Predictions saved to 'updated_fc_predictions_with_fc_hat.xlsx'.")


/Users/zhangtianjie/opt/anaconda3/lib/python3.8/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


Predictions saved to 'updated_fc_predictions_with_fc_hat.xlsx'.
